# 💻 Chapter 16: Bayesian Updating in Practice
*Track 2 — Developers | Tier 2*

> **🎬 Hook:** Machine learning models retrain on new data in batch. Bayesian updating does it online — one data point at a time, no retraining.

**🎯 Objectives:** Implement sequential Bayesian updating; understand conjugate priors; build an online learning system.

## 📖 Section 1 — Concept Review

### Bayesian Updating
Start with a prior → observe data → get posterior → posterior becomes the next prior.

$$\text{Posterior} \propto \text{Likelihood} \times \text{Prior}$$

### Conjugate Priors (closed-form updates)
| Data Distribution | Prior | Posterior |
|---|---|---|
| Bernoulli/Binomial | Beta(α,β) | Beta(α+successes, β+failures) |
| Poisson | Gamma(α,β) | Gamma(α+sum, β+n) |
| Normal (known σ) | Normal(μ₀,σ₀) | Normal(μₙ,σₙ) — weighted average |

### Why Conjugate?
Posterior is same distribution family as prior → closed-form math, no MCMC needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta, gamma, norm
import seaborn as sns
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# ── Beta-Binomial: Click-Through Rate ──
true_ctr = 0.073  # true CTR we're estimating
n_obs = [0, 5, 20, 100, 500, 2000]
clicks_by_n = {0: 0}
cumulative_n = 0
cumulative_clicks = 0

# Simulate streaming clicks
all_data = rng.random(2000) < true_ctr
click_counts = {0: 0}
total_by_n = {0: 0}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

alpha_prior, beta_prior = 1, 1  # uniform prior

for ax, n in zip(axes.flatten(), n_obs):
    x = np.linspace(0, 0.2, 300)
    if n == 0:
        posterior = beta(alpha_prior, beta_prior)
        title = 'Prior: Beta(1,1)
= Uniform[0,1]'
    else:
        successes = all_data[:n].sum()
        failures  = n - successes
        posterior = beta(alpha_prior + successes, beta_prior + failures)
        title = f'After n={n}: {successes} clicks
Beta({alpha_prior+successes},{beta_prior+failures})'

    ax.plot(x, posterior.pdf(x), lw=3, color='#3498db')
    ax.fill_between(x, posterior.pdf(x), alpha=0.3, color='#3498db')
    ax.axvline(true_ctr, color='red', lw=2, linestyle='--', label=f'True CTR={true_ctr}')
    if n > 0:
        ax.axvline(all_data[:n].mean(), color='green', lw=2, linestyle=':', label=f'MLE={all_data[:n].mean():.3f}')
    ax.set_title(title, fontweight='bold', fontsize=9)
    ax.set_xlabel('CTR'); ax.set_ylabel('Posterior Density')
    ax.legend(fontsize=7)
    ax.set_xlim(0, 0.2)

plt.suptitle("Bayesian Updating: Beta-Binomial for Click-Through Rate", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ch16_bayesian_updating.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Online Bayesian System: Website Anomaly Detection ──
rng = np.random.default_rng(42)

# Normal requests per minute, but with a DDoS starting at t=50
baseline_rate = 100
ddos_multiplier = 5

data_stream = np.concatenate([
    rng.poisson(baseline_rate, 50),
    rng.poisson(baseline_rate * ddos_multiplier, 50),
])

# Bayesian estimate of current rate (Gamma-Poisson conjugate)
# Prior: Gamma(alpha=5, beta=0.05) → prior mean = 100
alpha, beta_param = 5, 0.05

estimates = []
upper_bounds = []

for t, count in enumerate(data_stream):
    # Update: Gamma posterior after one Poisson observation
    alpha += count
    beta_param += 1
    mean_est = alpha / beta_param
    # 95th percentile of posterior
    upper = gamma.ppf(0.95, a=alpha, scale=1/beta_param)
    estimates.append(mean_est)
    upper_bounds.append(upper)

threshold = baseline_rate * 2.5

fig, ax = plt.subplots(figsize=(12, 5))
t = np.arange(100)
ax.plot(t, data_stream, alpha=0.5, color='gray', label='Observed requests/min')
ax.plot(t, estimates, color='#3498db', lw=2.5, label='Bayesian estimate')
ax.fill_between(t, estimates, upper_bounds, alpha=0.2, color='#3498db')
ax.axhline(threshold, color='#e74c3c', lw=2, linestyle='--', label=f'Alert threshold ({threshold})')
ax.axvline(50, color='orange', lw=2, linestyle=':', label='DDoS starts here')
ax.set_xlabel('Time (minutes)'); ax.set_ylabel('Requests per minute')
ax.set_title('🚨 Online Bayesian Anomaly Detection', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('ch16_anomaly.png', dpi=150, bbox_inches='tight')
plt.show()

## ✏️ Section 6 — Coding Challenges

**Challenge:** Implement a multi-armed bandit using Thompson Sampling.
You have 3 buttons (arms) with unknown click rates. At each step, sample from each arm's Beta posterior and select the arm with the highest sample.

<details><summary>Solution</summary>
See code below — Thompson Sampling naturally balances exploration vs exploitation.
</details>

In [ ]:
# Thompson Sampling Multi-Armed Bandit
rng = np.random.default_rng(42)

true_rates = [0.08, 0.12, 0.10]  # true CTRs for each arm
n_arms = len(true_rates)
n_rounds = 1000

alphas = np.ones(n_arms)  # Beta prior alpha
betas  = np.ones(n_arms)  # Beta prior beta
choices = []; rewards = []

for _ in range(n_rounds):
    # Thompson sampling: sample from each arm's posterior
    samples = [rng.beta(alphas[i], betas[i]) for i in range(n_arms)]
    chosen = np.argmax(samples)
    reward = int(rng.random() < true_rates[chosen])
    # Update posterior
    alphas[chosen] += reward
    betas[chosen]  += 1 - reward
    choices.append(chosen); rewards.append(reward)

print("🎰 Thompson Sampling Results:")
for i in range(n_arms):
    n_chosen = choices.count(i)
    est_rate = alphas[i] / (alphas[i] + betas[i])
    print(f"  Arm {i+1}: chosen {n_chosen} times, estimated rate={est_rate:.3f} (true={true_rates[i]})")

# Cumulative regret
optimal_rate = max(true_rates)
cum_regret = np.cumsum([optimal_rate - true_rates[c] for c in choices])
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(cum_regret, color='#e74c3c', lw=2.5)
ax.set_xlabel('Round'); ax.set_ylabel('Cumulative Regret')
ax.set_title('Thompson Sampling: Regret grows sub-linearly ✅', fontweight='bold')
plt.tight_layout()
plt.savefig('ch16_thompson.png', dpi=150, bbox_inches='tight')
plt.show()

## 🎯 Recap
1. Bayesian updating: posterior = f(prior × likelihood), done analytically with conjugate priors.
2. Beta-Binomial is the go-to for binary outcomes (clicks, conversions).
3. Online updating enables real-time systems without batch retraining.

**Next:** [Chapter 17 — Log Probabilities & Numerical Stability]